
# Example 3 — Cylinder target variance collapse on $\mathbb T^2$

This notebook builds a **variance-control example** using the **cylinder target itself**.

The key design choice is to pin a center
\[c=(1/2,1/2)\in \mathbb T^2,
\]
and use the smooth periodic spread observable
\[
q_c(x,y)=1-\frac12\Big(\cos(2\pi(x-c_x))+\cos(2\pi(y-c_y))\Big).
\]
For points near \(c\),
\[
q_c(x,y)\approx \pi^2 \,\|(x,y)-c\|^2,
\]
so the collective score
\[
F(\mu)=\langle q_c,\mu\rangle=\sum_{i=1}^n s_i q_c(x_i)
\]
acts like a **periodic variance proxy** around the pinned center.

We then use the cylinder terminal functional
\[
g(\mu)=\exp\!\left(-\frac{\lambda}{2}\big(F(\mu)-a\big)^2\right),
\]
with a **small target** \(a\), so the dynamics are encouraged to reduce the spread.

For a clear and seed-stable demo, I use the **direct cylinder drift**
\[
b_i = 2 D_i \log g
    = -2\lambda\big(F(\mu)-a\big)\nabla q_c(x_i),
\]
which is the short-horizon / terminal-drift version of the cylinder mechanism.
This is much more robust for variance collapse than the full quadrature heat solve, while still staying squarely in the **cylinder-\(g\)** setting.

Why this is stable across seeds:
- the objective is **collective** (only the global spread matters),
- the center is **pinned**, so the cluster does not drift to an arbitrary location,
- the observable is **smooth and periodic**,
- the target \(a\) is small but not exactly \(0\), which avoids over-forcing a single point.

The visualization reuses the same Plotly slider / play-pause style as the house$\to$flower notebook.


In [ ]:
import numpy as np
import sys
from dataclasses import dataclass
from pathlib import Path


ROOT_CANDIDATES = (Path.cwd().resolve(), *Path.cwd().resolve().parents, Path("/mnt/data").resolve())
ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "wasserstein_conditioning_algorithms.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate wasserstein_conditioning_algorithms.py")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from wasserstein_conditioning_algorithms import shortest_periodic_displacement, wrap_torus

np.set_printoptions(precision=3, suppress=True)


In [ ]:
import plotly.graph_objects as go

from notebooks.support import (
    center_trace,
    circle_trace as _circle_trace,
    configure_plotly,
    line_trace as _line_trace,
    make_particle_animation as _make_particle_animation,
)

configure_plotly()


def line_trace(points, name, color="rgba(80,80,80,0.55)", dash="dot", close=False, showlegend=True, marker_size=6):
    return _line_trace(
        points,
        name,
        color=color,
        dash=dash,
        close=close,
        showlegend=showlegend,
        marker_size=marker_size,
        mode="lines+markers",
    )



def circle_trace(center, radius=0.12, name="target region", color="rgba(80,80,80,0.45)", showlegend=True):
    return _circle_trace(center, radius=radius, name=name, color=color, showlegend=showlegend)



def make_particle_animation(
    positions,
    times,
    masses,
    title,
    static_traces=None,
    marker_size=18,
    x_range=(0.0, 1.0),
    y_range=(0.0, 1.0),
):
    return _make_particle_animation(
        positions,
        times,
        masses,
        title,
        static_traces=static_traces,
        marker_size=marker_size,
        x_range=x_range,
        y_range=y_range,
        mass_format=".3f",
        time_formatter=lambda t, h: f"time = {t:.4f}",
        slider_label_formatter=lambda t, h: f"{t:.4f}",
        currentvalue_prefix="time = ",
        width=800,
        height=700,
        play_frame_duration=85,
    )


In [ ]:

@dataclass(frozen=True)
class SimpleSimulation:
    times: np.ndarray
    positions: np.ndarray
    masses: np.ndarray
    drifts: np.ndarray | None = None

def ring_positions(n, center=(0.5, 0.5), radius=0.26, wobble=0.12):
    theta = np.linspace(0.0, 2.0 * np.pi, n, endpoint=False)
    rr = radius * (1.0 + wobble * np.cos(3.0 * theta))
    x = center[0] + rr * np.cos(theta)
    y = center[1] + rr * np.sin(theta)
    return wrap_torus(np.stack([x, y], axis=1))

def periodic_variance_proxy(center=(0.5, 0.5)):
    center = np.asarray(center, dtype=float)

    def observable(points):
        pts = np.asarray(points, dtype=float)
        dx = pts[..., 0] - center[0]
        dy = pts[..., 1] - center[1]
        return 1.0 - 0.5 * (np.cos(2.0 * np.pi * dx) + np.cos(2.0 * np.pi * dy))

    def gradient(points):
        pts = np.asarray(points, dtype=float)
        dx = pts[..., 0] - center[0]
        dy = pts[..., 1] - center[1]
        return np.stack(
            [
                np.pi * np.sin(2.0 * np.pi * dx),
                np.pi * np.sin(2.0 * np.pi * dy),
            ],
            axis=-1,
        )

    return observable, gradient

def simulate_direct_cylinder_variance_collapse(
    masses,
    initial_positions,
    observable,
    grad_observable,
    target_spread,
    lambda_,
    horizon,
    step_size,
    *,
    rng=None,
    store_drifts=True,
):
    masses = np.asarray(masses, dtype=float).reshape(-1)
    masses = masses / masses.sum()
    x = wrap_torus(np.asarray(initial_positions, dtype=float))
    n, d = x.shape

    steps_float = horizon / step_size
    steps = int(round(steps_float))
    if not np.isclose(steps_float, steps):
        raise ValueError("horizon / step_size must be an integer")

    rng = np.random.default_rng() if rng is None else rng
    times = np.linspace(0.0, horizon, steps + 1)
    positions = np.empty((steps + 1, n, d), dtype=float)
    positions[0] = x
    drifts = np.empty((steps, n, d), dtype=float) if store_drifts else None
    noise_scale = np.sqrt(2.0 * step_size / masses)[:, None]

    for m in range(steps):
        collective_spread = float(np.sum(masses * observable(x)))
        drift = -2.0 * lambda_ * (collective_spread - target_spread) * grad_observable(x)
        x = wrap_torus(x + drift * step_size + noise_scale * rng.normal(size=(n, d)))
        positions[m + 1] = x
        if drifts is not None:
            drifts[m] = drift

    return SimpleSimulation(times=times, positions=positions, masses=masses, drifts=drifts)

def simulate_free_diffusion(
    masses,
    initial_positions,
    horizon,
    step_size,
    *,
    rng=None,
):
    masses = np.asarray(masses, dtype=float).reshape(-1)
    masses = masses / masses.sum()
    x = wrap_torus(np.asarray(initial_positions, dtype=float))
    n, d = x.shape

    steps_float = horizon / step_size
    steps = int(round(steps_float))
    if not np.isclose(steps_float, steps):
        raise ValueError("horizon / step_size must be an integer")

    rng = np.random.default_rng() if rng is None else rng
    times = np.linspace(0.0, horizon, steps + 1)
    positions = np.empty((steps + 1, n, d), dtype=float)
    positions[0] = x
    noise_scale = np.sqrt(2.0 * step_size / masses)[:, None]

    for m in range(steps):
        x = wrap_torus(x + noise_scale * rng.normal(size=(n, d)))
        positions[m + 1] = x

    return SimpleSimulation(times=times, positions=positions, masses=masses, drifts=None)

def weighted_variance_proxy(positions, masses, observable):
    return np.array([np.sum(masses * observable(pos)) for pos in positions], dtype=float)

def weighted_mean_squared_radius(positions, masses, center=(0.5, 0.5)):
    center = np.asarray(center, dtype=float)[None, :]
    displacement = shortest_periodic_displacement(positions, center)
    return np.sum(masses[None, :] * np.sum(displacement ** 2, axis=-1), axis=1)

def target_guide_radius_from_proxy(target_spread):
    # Near the center: q_c(x) ≈ π² ||x-c||²
    return float(np.sqrt(max(target_spread, 1e-12)) / np.pi)


In [ ]:

# --- collective variance-collapse setup ---
masses = np.array([0.30, 0.24, 0.18, 0.12, 0.09, 0.07], dtype=float)
masses = masses / masses.sum()

center = np.array([0.5, 0.5], dtype=float)
initial_positions = ring_positions(len(masses), center=center, radius=0.26, wobble=0.12)

observable, grad_observable = periodic_variance_proxy(center=center)

# Small target spread -> low terminal variance, but not exactly zero.
target_spread = 0.01
lambda_ = 200.0
horizon = 0.02
steps = 200
step_size = horizon / steps

seed = 2
robustness_seeds = list(range(8))

sim = simulate_direct_cylinder_variance_collapse(
    masses=masses,
    initial_positions=initial_positions,
    observable=observable,
    grad_observable=grad_observable,
    target_spread=target_spread,
    lambda_=lambda_,
    horizon=horizon,
    step_size=step_size,
    rng=np.random.default_rng(seed),
    store_drifts=True,
)

free_sim = simulate_free_diffusion(
    masses=masses,
    initial_positions=initial_positions,
    horizon=horizon,
    step_size=step_size,
    rng=np.random.default_rng(seed),
)

spread_curve = weighted_variance_proxy(sim.positions, sim.masses, observable)
radius_curve = weighted_mean_squared_radius(sim.positions, sim.masses, center=center)

free_spread_curve = weighted_variance_proxy(free_sim.positions, free_sim.masses, observable)
free_radius_curve = weighted_mean_squared_radius(free_sim.positions, free_sim.masses, center=center)

guide_radius = max(0.11, 2.5 * target_guide_radius_from_proxy(target_spread))

print("masses:", masses)
print("target spread a:", target_spread)
print("lambda:", lambda_, "horizon:", horizon, "step size:", step_size, "steps:", steps, "seed:", seed)
print("initial variance proxy:", float(spread_curve[0]))
print("final variance proxy:", float(spread_curve[-1]))
print("initial weighted mean-squared radius:", float(radius_curve[0]))
print("final weighted mean-squared radius:", float(radius_curve[-1]))
print("free terminal weighted mean-squared radius (same seed):", float(free_radius_curve[-1]))


In [ ]:

static_traces = [
    line_trace(initial_positions, name="initial ring guide", color="rgba(30, 144, 255, 0.45)", dash="dash", close=True),
    circle_trace(center, radius=guide_radius, name="low-variance guide", color="rgba(220, 20, 60, 0.50)"),
    center_trace(np.array([center]), name="pinned center c", color="rgba(220, 20, 60, 0.80)", symbol="x", size=12),
]

fig = make_particle_animation(
    positions=sim.positions,
    times=sim.times,
    masses=sim.masses,
    title="Example 3: cylinder target variance collapse (direct cylinder drift)",
    static_traces=static_traces,
    marker_size=20,
)
fig.show()


In [ ]:

comparison_fig = go.Figure()

comparison_fig.add_trace(circle_trace(center, radius=guide_radius, name="low-variance guide", color="rgba(220, 20, 60, 0.50)"))
comparison_fig.add_trace(center_trace(np.array([center]), name="pinned center c", color="rgba(220, 20, 60, 0.80)", symbol="x", size=12))

comparison_fig.add_trace(
    go.Scatter(
        x=initial_positions[:, 0],
        y=initial_positions[:, 1],
        mode="markers",
        name="initial positions",
        marker=dict(size=14, symbol="circle-open", color="rgba(30, 144, 255, 0.85)", line=dict(color="rgba(30, 144, 255, 0.85)", width=2)),
    )
)

comparison_fig.add_trace(
    go.Scatter(
        x=free_sim.positions[-1, :, 0],
        y=free_sim.positions[-1, :, 1],
        mode="markers",
        name="free diffusion terminal positions",
        marker=dict(size=13, symbol="x", color="rgba(110, 110, 110, 0.90)", line=dict(color="rgba(110, 110, 110, 0.90)", width=2)),
    )
)

comparison_fig.add_trace(
    go.Scatter(
        x=sim.positions[-1, :, 0],
        y=sim.positions[-1, :, 1],
        mode="markers",
        name="conditioned terminal positions",
        marker=dict(
            size=16,
            color=sim.masses,
            colorscale="Viridis",
            cmin=float(np.min(sim.masses)),
            cmax=float(np.max(sim.masses)),
            showscale=False,
            line=dict(color="black", width=1),
        ),
        text=[f"particle {i}<br>mass = {m:.3f}" for i, m in enumerate(sim.masses)],
        hovertemplate="%{text}<br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
    )
)

comparison_fig.update_layout(
    title="Initial vs terminal positions: conditioned collapse compared with free diffusion",
    template="simple_white",
    width=800,
    height=680,
    xaxis=dict(range=[0.0, 1.0], title="x", scaleanchor="y", scaleratio=1),
    yaxis=dict(range=[0.0, 1.0], title="y"),
    legend=dict(x=0.01, y=0.99),
)
comparison_fig.show()


In [ ]:

spread_fig = go.Figure()
spread_fig.add_trace(
    go.Scatter(
        x=sim.times,
        y=spread_curve,
        mode="lines",
        name="conditioned",
    )
)
spread_fig.add_trace(
    go.Scatter(
        x=free_sim.times,
        y=free_spread_curve,
        mode="lines",
        name="free diffusion",
        opacity=0.8,
    )
)
spread_fig.add_hline(
    y=target_spread,
    line_dash="dash",
    annotation_text="target a",
    annotation_position="top left",
)
spread_fig.update_layout(
    title="Periodic variance proxy over time",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title=r"$\mu_t(q_c)$",
)
spread_fig.show()

radius_fig = go.Figure()
radius_fig.add_trace(
    go.Scatter(
        x=sim.times,
        y=radius_curve,
        mode="lines",
        name="conditioned",
    )
)
radius_fig.add_trace(
    go.Scatter(
        x=free_sim.times,
        y=free_radius_curve,
        mode="lines",
        name="free diffusion",
        opacity=0.8,
    )
)
radius_fig.update_layout(
    title="Weighted mean-squared periodic radius over time",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted mean-squared radius to c",
)
radius_fig.show()

print("conditioned terminal proxy:", float(spread_curve[-1]))
print("free terminal proxy:", float(free_spread_curve[-1]))
print("conditioned terminal weighted mean-squared radius:", float(radius_curve[-1]))
print("free terminal weighted mean-squared radius:", float(free_radius_curve[-1]))



### Seed robustness check

The next cell reruns the **same pinned variance-collapse setup** for several seeds.

A pure variance objective without a pinned center would be translation-invariant, so the final cluster could land anywhere on the torus and the visual result would not be seed-stable.
Pinning the center \(c\) removes that symmetry and makes the terminal behavior much more consistent.


In [ ]:

seed_to_spread = {}
seed_to_radius = {}
seed_to_final_positions = {}

for s in robustness_seeds:
    sim_s = simulate_direct_cylinder_variance_collapse(
        masses=masses,
        initial_positions=initial_positions,
        observable=observable,
        grad_observable=grad_observable,
        target_spread=target_spread,
        lambda_=lambda_,
        horizon=horizon,
        step_size=step_size,
        rng=np.random.default_rng(s),
        store_drifts=False,
    )
    seed_to_spread[s] = weighted_variance_proxy(sim_s.positions, sim_s.masses, observable)
    seed_to_radius[s] = weighted_mean_squared_radius(sim_s.positions, sim_s.masses, center=center)
    seed_to_final_positions[s] = sim_s.positions[-1]

final_spreads = np.array([seed_to_spread[s][-1] for s in robustness_seeds], dtype=float)
final_radii = np.array([seed_to_radius[s][-1] for s in robustness_seeds], dtype=float)

robust_fig = go.Figure()
for s in robustness_seeds:
    robust_fig.add_trace(
        go.Scatter(
            x=sim.times,
            y=seed_to_radius[s],
            mode="lines",
            name=f"seed {s}",
            opacity=0.75,
        )
    )

robust_fig.update_layout(
    title="Seed robustness: weighted mean-squared radius across several runs",
    template="simple_white",
    width=760,
    height=420,
    xaxis_title="time",
    yaxis_title="weighted mean-squared radius to c",
)
robust_fig.show()

terminal_overlay = go.Figure()
terminal_overlay.add_trace(circle_trace(center, radius=guide_radius, name="low-variance guide", color="rgba(220, 20, 60, 0.50)"))
terminal_overlay.add_trace(center_trace(np.array([center]), name="pinned center c", color="rgba(220, 20, 60, 0.80)", symbol="x", size=12))

for s in robustness_seeds:
    pos = seed_to_final_positions[s]
    terminal_overlay.add_trace(
        go.Scatter(
            x=pos[:, 0],
            y=pos[:, 1],
            mode="markers",
            name=f"seed {s}",
            marker=dict(size=11, symbol="circle-open"),
            opacity=0.70,
            showlegend=True,
        )
    )

terminal_overlay.update_layout(
    title="Terminal positions for several seeds",
    template="simple_white",
    width=800,
    height=680,
    xaxis=dict(range=[0.0, 1.0], title="x", scaleanchor="y", scaleratio=1),
    yaxis=dict(range=[0.0, 1.0], title="y"),
    legend=dict(x=0.01, y=0.99),
)
terminal_overlay.show()

print("terminal variance proxy across seeds:")
print("  mean =", float(final_spreads.mean()))
print("  std  =", float(final_spreads.std()))
print("  min  =", float(final_spreads.min()))
print("  max  =", float(final_spreads.max()))

print("terminal weighted mean-squared radius across seeds:")
print("  mean =", float(final_radii.mean()))
print("  std  =", float(final_radii.std()))
print("  min  =", float(final_radii.min()))
print("  max  =", float(final_radii.max()))



### What to tweak

The most useful knobs are:

- **`target_spread`**: smaller means tighter final concentration.
- **`lambda_`**: larger means stronger collective correction.
- **`horizon`**: if it is too long, diffusion has more time to re-spread after the cluster forms.
- **masses / particle count**: fewer, heavier particles reduce diffusion noise and make the collapse cleaner.

For this notebook, the current setting was chosen because it gives a visibly clear collapse and stays fairly concentrated across seeds.
